# Chapter 4: Unicode Text Versus Bytes
## Fluent Python, 2nd Edition -- Distilled Interactive Notebook

> "Humans use text. Computers speak bytes."
> -- *Esther Nam and Travis Fischer*

Related wiki articles: [[characters-and-bytes]], [[encoding-and-decoding]], [[text-file-handling]], [[unicode-normalization]], [[case-folding-and-text-utils]], [[sorting-unicode]], [[unicode-database-and-dual-mode]]

## TL;DR

- **str** holds Unicode code points (abstract text); **bytes/bytearray** hold raw octets. Encoding converts str to bytes; decoding converts bytes to str.
- Always pass `encoding=` to `open()`. The **Unicode sandwich** pattern: decode on input, work with str, encode on output.
- **UnicodeEncodeError** and **UnicodeDecodeError** arise from codec mismatches. Use error handlers (`strict`, `ignore`, `replace`, `xmlcharrefreplace`) when needed.
- **Normalization** (NFC/NFD/NFKC/NFKD) is essential for reliable text comparison: `"cafe\u0301" != "caf\u00e9"` even though they look identical.
- Use `str.casefold()` (not `.lower()`) for case-insensitive comparisons. Use `unicodedata` to query character metadata.

---
## 1. Characters vs Bytes

A **character** is a Unicode code point (U+0000 to U+10FFFF). A **byte** is an integer 0-255. **Encoding** maps code points to bytes; **decoding** maps bytes back to code points.

See also: [[characters-and-bytes]]

In [ ]:
# str vs bytes: the fundamental distinction
s = 'cafe\u0301'  # e + combining acute accent
print(f"str: {s!r}  display: {s}  len: {len(s)}")

# Encode to bytes
b = s.encode('utf-8')
print(f"bytes: {b!r}  len: {len(b)}")

# Decode back
s2 = b.decode('utf-8')
print(f"decoded: {s2!r}  display: {s2}")

In [ ]:
# bytes and bytearray basics
cafe = bytes('caf\u00e9', encoding='utf_8')
print("bytes repr:", cafe)
print("First item (int):", cafe[0])    # integer
print("First slice (bytes):", cafe[:1]) # bytes

# bytearray is mutable bytes
ba = bytearray(cafe)
print("bytearray:", ba)
ba[0] = 0x43  # Change 'c' to 'C'
print("modified:", ba, "->", ba.decode('utf-8'))

In [ ]:
# bytes from hex and from arrays
raw = bytes.fromhex('31 4B CE A9')
print("From hex:", raw, "->", raw.decode('utf-8'))

import array
numbers = array.array('h', [-2, -1, 0, 1, 2])  # short ints
octets = bytes(numbers)
print(f"Array {numbers} as bytes: {octets!r} ({len(octets)} bytes)")

---
## 2. Encoding and Decoding

Python bundles 100+ codecs. The key ones: **UTF-8** (dominant, variable-length, ASCII-compatible), **latin-1** (basis for many Western encodings), **UTF-16** (uses BOM for endianness).

See also: [[encoding-and-decoding]]

In [ ]:
# Same string, different encodings -> different bytes
text = 'El Niño'
for codec in ['latin_1', 'utf_8', 'utf_16']:
    encoded = text.encode(codec)
    print(f"{codec:10} {encoded!r}")

In [ ]:
# UnicodeEncodeError: when a codec can't handle a character
city = 'São Paulo'  # a with tilde

# UTF-8 handles everything
print("utf-8:", city.encode('utf-8'))

# cp437 can't encode a-tilde
for handler in ['strict', 'ignore', 'replace', 'xmlcharrefreplace']:
    try:
        result = city.encode('cp437', errors=handler)
        print(f"cp437 ({handler:20}): {result}")
    except UnicodeEncodeError as e:
        print(f"cp437 ({handler:20}): UnicodeEncodeError - {e.reason}")

In [ ]:
# UnicodeDecodeError: wrong codec for the byte sequence
octets = b'Montr\xe9al'  # \xe9 is e-acute in latin-1

print("cp1252:", octets.decode('cp1252'))   # correct (superset of latin-1)
print("iso8859_7:", octets.decode('iso8859_7'))  # Greek: wrong char
print("koi8_r:", octets.decode('koi8_r'))    # Russian: wrong char

try:
    octets.decode('utf_8')
except UnicodeDecodeError as e:
    print(f"utf-8: UnicodeDecodeError - {e.reason}")

# replace handler shows replacement character
print("utf-8 (replace):", octets.decode('utf_8', errors='replace'))

---
## 3. Text File Handling: The Unicode Sandwich

**Best practice**: decode bytes to str on input, work with str internally, encode str to bytes on output. **Always** pass `encoding=` to `open()` -- never rely on platform defaults.

See also: [[text-file-handling]]

In [ ]:
import tempfile, os

# Write with explicit encoding
path = os.path.join(tempfile.gettempdir(), 'cafe_test.txt')
with open(path, 'w', encoding='utf-8') as f:
    chars_written = f.write('cafe\u0301')
    print(f"Wrote {chars_written} characters")

# Read with explicit encoding -- correct
with open(path, 'r', encoding='utf-8') as f:
    print("UTF-8 read:", repr(f.read()))

# Read in binary mode to see raw bytes
with open(path, 'rb') as f:
    raw = f.read()
    print(f"Raw bytes: {raw!r} ({len(raw)} bytes)")

# Check file size
print(f"File size: {os.stat(path).st_size} bytes")
os.unlink(path)  # cleanup

In [ ]:
# Encoding defaults (platform-specific -- why you should NEVER rely on them)
import sys, locale

print("locale preferred:", locale.getpreferredencoding())
print("sys.stdout.encoding:", sys.stdout.encoding)
print("sys.getdefaultencoding():", sys.getdefaultencoding())
print("sys.getfilesystemencoding():", sys.getfilesystemencoding())

---
## 4. Unicode Normalization (NFC / NFD / NFKC / NFKD)

The same visual character can have multiple Unicode representations. `"cafe\u0301"` (e + combining accent) looks identical to `"caf\u00e9"` (precomposed) but they are **not equal**. Normalization fixes this.

- **NFC** (compose): canonical composition -- the W3C recommendation
- **NFD** (decompose): canonical decomposition
- **NFKC/NFKD**: compatibility normalization (lossy -- use for search, not storage)

See also: [[unicode-normalization]]

In [ ]:
from unicodedata import normalize, name

# The fundamental problem: same appearance, different bytes
s1 = 'caf\u00e9'      # precomposed e-acute
s2 = 'cafe\u0301'     # e + combining acute accent

print(f"s1: {s1!r}  len={len(s1)}")
print(f"s2: {s2!r}  len={len(s2)}")
print(f"s1 == s2: {s1 == s2}")
print(f"Display identical: '{s1}' vs '{s2}'")

In [ ]:
# NFC and NFD fix the comparison problem
from unicodedata import normalize

s1 = 'caf\u00e9'
s2 = 'cafe\u0301'

# NFC: compose to shortest form
nfc1 = normalize('NFC', s1)
nfc2 = normalize('NFC', s2)
print(f"NFC: {nfc1!r} == {nfc2!r} -> {nfc1 == nfc2}")

# NFD: decompose to base + combining chars
nfd1 = normalize('NFD', s1)
nfd2 = normalize('NFD', s2)
print(f"NFD: {nfd1!r} == {nfd2!r} -> {nfd1 == nfd2}")

# Best practice: normalize to NFC for storage/comparison
print(f"\nNFC length: {len(nfc1)}, NFD length: {len(nfd1)}")

In [ ]:
# NFKC/NFKD: compatibility normalization (lossy!)
from unicodedata import normalize

# micro sign vs greek mu
micro = '\u00b5'     # MICRO SIGN
mu = '\u03bc'        # GREEK SMALL LETTER MU
print(f"micro: {micro}  mu: {mu}")
print(f"Equal? {micro == mu}")
print(f"NFKC equal? {normalize('NFKC', micro) == normalize('NFKC', mu)}")

# Fractions
half = '\u00bd'  # VULGAR FRACTION ONE HALF
print(f"\nNFKC('{half}'): '{normalize('NFKC', half)}'")

# WARNING: NFKC/NFKD can lose information -- use for search, not storage
ohm = '\u2126'
print(f"NFC('{ohm}'): '{normalize('NFC', ohm)}' == OHM->OMEGA")

---
## 5. Case Folding and Text Utilities

`str.casefold()` goes beyond `.lower()` for case-insensitive comparisons. Combine with NFC normalization for reliable text matching.

See also: [[case-folding-and-text-utils]]

In [ ]:
# casefold vs lower -- casefold handles more edge cases
examples = [
    'Stra\u00dfe',       # German sharp s
    '\u00b5',            # micro sign
    '\ufb01',            # fi ligature
]
for s in examples:
    lo = s.lower()
    cf = s.casefold()
    print(f"'{s}' -> lower='{lo}' casefold='{cf}'  same={lo==cf}")

In [ ]:
# Utility function: NFC + casefold for reliable comparison
from unicodedata import normalize

def nfc_equal(str1, str2):
    """Compare strings using NFC normalization."""
    return normalize('NFC', str1) == normalize('NFC', str2)

def fold_equal(str1, str2):
    """Case-insensitive comparison with NFC + casefold."""
    return (normalize('NFC', str1).casefold() ==
            normalize('NFC', str2).casefold())

# NFC comparison
s1 = 'caf\u00e9'
s2 = 'cafe\u0301'
print(f"nfc_equal('{s1}', '{s2}'): {nfc_equal(s1, s2)}")

# Case-insensitive comparison
strasse1 = 'Stra\u00dfe'
strasse2 = 'strasse'
result = fold_equal(strasse1, strasse2)
print(f"fold_equal('{strasse1}', '{strasse2}'): {result}")

---
## 6. Sorting Unicode Text

`sorted()` compares Unicode code points, **not** linguistic order. Accented characters sort incorrectly by default. Use `locale.strxfrm` or the `pyuca` library for proper sorting.

See also: [[sorting-unicode]]

In [ ]:
# Default sort is by code point -- wrong for accented chars
fruits = ['caju', 'a\u00e7a\u00ed', 'acerola', 'ma\u00e7\u00e3']
print("Display:", fruits)
print("Default sorted:", sorted(fruits))

# locale.strxfrm for locale-aware sorting
import locale
try:
    locale.setlocale(locale.LC_COLLATE, 'en_US.UTF-8')
    sorted_locale = sorted(fruits, key=locale.strxfrm)
    print("Locale sorted:", sorted_locale)
except locale.Error:
    print("(locale en_US.UTF-8 not available on this system)")

---
## 7. The Unicode Database and Dual-Mode APIs

The `unicodedata` module provides character metadata: name, category, numeric value. The `re` and `os` modules behave differently with `str` vs `bytes` arguments.

See also: [[unicode-database-and-dual-mode]]

In [ ]:
import unicodedata

# Explore character metadata
chars = ['A', 'é', 'Ω', '½', '☃', 'µ']
for c in chars:
    print(f"  U+{ord(c):04X}  {c}  name={unicodedata.name(c, '???'):30}  "
          f"cat={unicodedata.category(c)}")

In [ ]:
# Finding characters by name
import unicodedata, sys

def find_chars(query, start=32, end=0x10000):
    """Search Unicode database for characters matching query."""
    query = query.upper()
    results = []
    for code in range(start, min(end, sys.maxunicode + 1)):
        char = chr(code)
        try:
            char_name = unicodedata.name(char)
        except ValueError:
            continue
        if query in char_name:
            results.append((code, char, char_name))
    return results

# Find chess pieces
for code, char, char_name in find_chars('CHESS', end=0x1FA00):
    print(f"  U+{code:04X}  {char}  {char_name}")

In [ ]:
# Dual-mode: str vs bytes in regex
import re

# str pattern matches Unicode letters
str_pattern = re.compile(r'\w+')
text = 'Guten Tag, caf\u00e9!'
print("str matches:", str_pattern.findall(text))

# bytes pattern matches only ASCII word chars
bytes_pattern = re.compile(rb'\w+')
print("bytes matches:", bytes_pattern.findall(text.encode('utf-8')))

---
## Exercises

Test your understanding of Chapter 4 concepts.

### Exercise 1: Encode/Decode Round-Trip
Encode the string `"Hello, 世界!"` ("Hello, World!" with Chinese) to UTF-8, then decode it back. Verify the round-trip is lossless.

In [ ]:
# Exercise 1
text = "Hello, 世界!"
print("Original:", text)

# Solution:
encoded = text.encode('utf-8')
print("Encoded:", encoded, f"({len(encoded)} bytes)")

decoded = encoded.decode('utf-8')
print("Decoded:", decoded)
print("Round-trip lossless:", text == decoded)

### Exercise 2: Normalization
Show that the two representations of "naïve" (with combining diaeresis) and "naïve" (precomposed) are NFC-equal but not == equal.

In [ ]:
# Exercise 2
from unicodedata import normalize

s1 = 'nai\u0308ve'    # i + combining diaeresis
s2 = 'na\u00efve'     # precomposed i-diaeresis

# Solution:
print(f"s1: {s1!r} -> display: {s1}")
print(f"s2: {s2!r} -> display: {s2}")
print(f"s1 == s2: {s1 == s2}")
print(f"NFC equal: {normalize('NFC', s1) == normalize('NFC', s2)}")

### Exercise 3: Character Detective
Use `unicodedata` to find the name and category of these characters: ©, ☃, ὠ0 (if supported), ∞

In [ ]:
# Exercise 3
import unicodedata

# Solution:
for char in ['©', '☃', '😀', '∞']:
    try:
        char_name = unicodedata.name(char)
    except ValueError:
        char_name = '(no name)'
    print(f"  U+{ord(char):04X}  {char}  {char_name}  category={unicodedata.category(char)}")

---
## Key Takeaways

1. **str = Unicode text, bytes = raw data.** Encoding converts str to bytes; decoding converts bytes to str.
2. **Always pass `encoding=` to `open()`.** Never rely on platform defaults -- they vary across machines and locales.
3. **Unicode sandwich**: decode bytes on input, process str internally, encode on output.
4. **NFC normalization** is essential for reliable string comparison. Two strings can look identical but be `!=`.
5. **casefold()** is stronger than **lower()** for case-insensitive matching (handles German sharp-s, micro sign, etc.).
6. **NFKC/NFKD** are lossy -- use for search/indexing only, never for canonical storage.
7. **str vs bytes in APIs**: regex `\w` matches Unicode letters in str mode but only ASCII in bytes mode.
8. **unicodedata** is your explorer tool for the Unicode database -- names, categories, numeric values.

See also: [[characters-and-bytes]], [[encoding-and-decoding]], [[text-file-handling]], [[unicode-normalization]], [[case-folding-and-text-utils]], [[sorting-unicode]], [[unicode-database-and-dual-mode]]